# Feature Engineering & Preprocessing Pipeline

A reusable, leak-free Scikit-Learn preprocessing workflow for a complex tabular classification dataset.

## Objectives
- Load a mixed numerical + categorical tabular dataset.
- Split into train/test **before fitting any transformation**.
- Impute missing values inside a reusable pipeline.
- Scale numerical features.
- One-hot encode categorical features.
- Analyze numerical correlations using training data only.
- Rank features using tree-based feature importance.
- Evaluate the complete pipeline on an untouched test set.

**Dataset:** Adult Census Income dataset from OpenML.  
**Target:** Whether annual income is `>50K`.

In [ ]:
# Install dependencies if needed:
# %pip install -q pandas numpy scikit-learn matplotlib seaborn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

## 1. Load the dataset

The Adult dataset contains both numerical and categorical columns and includes missing values represented by `?`. These characteristics make it suitable for demonstrating real-world preprocessing.

In [ ]:
adult = fetch_openml(name="adult", version=2, as_frame=True)
df = adult.frame.copy()

print("Dataset shape:", df.shape)
display(df.head())
display(df.info())

In [ ]:
# Standardize missing-value representation.
# OpenML may already represent missing values as NaN, but this also handles '?' safely.
df = df.replace("?", np.nan)

print("Missing values by column:")
display(df.isna().sum().sort_values(ascending=False).head(15))

print("\nTarget distribution:")
display(df["class"].value_counts(dropna=False))

## 2. Separate features and target

The target column is separated before preprocessing. No imputer, scaler, encoder, or feature-selection step is fitted on the complete dataset.

In [ ]:
target_column = "class"

X = df.drop(columns=[target_column])
y = df[target_column].astype(str).str.strip()

# Train/test split happens BEFORE any transformation.
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

## 3. Identify numerical and categorical features

In [ ]:
numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number"]).columns.tolist()

print("Numerical features:", numeric_features)
print("\nCategorical features:", categorical_features)

## 4. Build leak-free preprocessing pipelines

### Numerical pipeline
1. Median imputation
2. Standard scaling

### Categorical pipeline
1. Most-frequent imputation
2. One-hot encoding

The transformers are fitted only through `X_train` when the final pipeline is trained.

In [ ]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=5))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ],
    remainder="drop"
)

## 5. Correlation analysis

Correlation is calculated using **training data only** so that the test set remains untouched.

A heatmap is useful for detecting strongly correlated numerical variables and possible redundancy.

In [ ]:
corr = X_train[numeric_features].corr(numeric_only=True)

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Numerical Feature Correlation — Training Data Only")
plt.tight_layout()
plt.show()

## 6. Create the complete ML pipeline

The preprocessing stage and Random Forest classifier are combined into one reusable Scikit-Learn `Pipeline`.

This prevents a common data-leakage problem: accidentally preprocessing the test set using statistics learned from the entire dataset.

In [ ]:
model = RandomForestClassifier(
    n_estimators=250,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced_subsample"
)

model_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", model)
])

model_pipeline.fit(X_train, y_train)
print("Pipeline training completed.")

## 7. Evaluate on the untouched test set

In [ ]:
y_pred = model_pipeline.predict(X_test)
y_prob = model_pipeline.predict_proba(X_test)[:, 1]

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("ROC-AUC:", round(roc_auc_score((y_test == ">50K").astype(int), y_prob), 4))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

## 8. Tree-based feature importance

The Random Forest is fitted after preprocessing. We extract the transformed feature names and map them to the model's feature-importance values.

This shows which original/encoded variables contributed most strongly to the trained model.

In [ ]:
feature_names = model_pipeline.named_steps["preprocessor"].get_feature_names_out()
importances = model_pipeline.named_steps["model"].feature_importances_

feature_importance = (
    pd.DataFrame({
        "feature": feature_names,
        "importance": importances
    })
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

display(feature_importance.head(20))

In [ ]:
top_n = 20
top_features = feature_importance.head(top_n).sort_values("importance")

plt.figure(figsize=(10, 8))
plt.barh(top_features["feature"], top_features["importance"])
plt.xlabel("Random Forest Importance")
plt.ylabel("Transformed Feature")
plt.title(f"Top {top_n} Features")
plt.tight_layout()
plt.show()

## 9. Optional mutual-information analysis

Mutual information can measure dependency between a feature and the target. For a mixed tabular dataset, calculating it after one-hot encoding is straightforward, although the resulting ranking is at the encoded-feature level.

The analysis below is performed using the training data only.

In [ ]:
from sklearn.feature_selection import mutual_info_classif

X_train_transformed = model_pipeline.named_steps["preprocessor"].transform(X_train)
mi_target = (y_train == ">50K").astype(int).to_numpy()

mi_scores = mutual_info_classif(
    X_train_transformed,
    mi_target,
    random_state=42
)

mutual_info = (
    pd.DataFrame({
        "feature": feature_names,
        "mutual_information": mi_scores
    })
    .sort_values("mutual_information", ascending=False)
    .reset_index(drop=True)
)

display(mutual_info.head(20))

## 10. Final leakage checks

The following checks document why this workflow is considered leak-free:

- The test set was created before preprocessing.
- Imputation statistics are learned from training data only.
- Scaling parameters are learned from training data only.
- One-hot categories are learned from training data only.
- The Random Forest is trained only on transformed training data.
- Feature-importance and mutual-information analysis use training data.
- The final test set is used only for evaluation.

In [ ]:
print("Leakage-safe workflow checklist")
checks = {
    "Train/test split before transformations": True,
    "Numerical imputation inside Pipeline": True,
    "Numerical scaling inside Pipeline": True,
    "Categorical imputation inside Pipeline": True,
    "One-hot encoding inside Pipeline": True,
    "Feature analysis based on training data": True,
    "Test set reserved for final evaluation": True,
}
for item, status in checks.items():
    print(f"[{'PASS' if status else 'FAIL'}] {item}")

## Conclusion

This project demonstrates a reusable preprocessing pipeline for real-world tabular machine learning. It handles missing values, numerical scaling, categorical encoding, correlation analysis, feature ranking, model training, and evaluation while keeping preprocessing isolated from the test set to minimize data leakage.

### Key Scikit-Learn concepts demonstrated
- `train_test_split`
- `SimpleImputer`
- `StandardScaler`
- `OneHotEncoder`
- `ColumnTransformer`
- `Pipeline`
- `RandomForestClassifier`
- Feature importance
- Mutual information
- Classification metrics